In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/miadul/hospital-readmission-risk-dataset/hospital_readmission_risk_10000.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/miadul/hospital-readmission-risk-dataset/hospital_readmission_risk_10000.csv')
df = df.drop('patient_id',axis=1)
df = df.dropna()
df.head()

,age,gender,weight_kg,height_cm,bmi,num_previous_admissions,chronic_conditions,medications_count,last_hemoglobin,last_glucose,...,length_of_stay,procedures_count,smoking_status,alcohol_use,physical_activity,insurance_type,followup_compliance,social_support,mental_health_issue,readmission_risk
0,18,Male,57,157,23.1,8,Heart Disease,8,16.2,135.1,...,7,0,Current,Moderate,Medium,Private,Poor,Weak,Yes,Medium
1,23,Female,117,150,52.0,6,Diabetes,6,13.1,137.2,...,21,2,Never,High,Medium,Private,Good,Weak,Yes,High
2,80,Female,61,141,30.7,5,Heart Disease,8,15.4,82.1,...,2,0,Never,Moderate,Low,Public,Good,Weak,No,Low
4,14,Male,67,179,20.9,6,Heart Disease,0,14.7,84.0,...,3,0,Former,High,High,Uninsured,Poor,Weak,No,Medium
5,74,Male,79,151,34.6,8,Heart Disease,2,10.6,93.5,...,22,4,Former,Moderate,Medium,Uninsured,Poor,Weak,No,High


In [3]:
from sklearn.preprocessing import LabelEncoder

In [4]:
le = LabelEncoder()
df['readmission_label'] = le.fit_transform(df['readmission_risk'])                  

In [5]:
y = df['readmission_label']
x = df.drop(columns = ['readmission_label','readmission_risk'])

In [6]:
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['readmission_label'] = df['readmission_risk'].map(risk_map)

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x,y,train_size=0.79,random_state=42)

In [9]:
from catboost import CatBoostClassifier

In [10]:
cat_features=[]
for i in x.columns:
    if x[i].dtype == 'object':
        cat_features.append(i)

In [11]:
cb = CatBoostClassifier(
    iterations=1000,           # ağaç sayısını arttırdım
    learning_rate=0.03,        # daha küçük bir adım boyu yaptım
    depth=6,                   
    loss_function='MultiClass',
    early_stopping_rounds=50,  # gelişme durduğunda eğitimi keser
    eval_metric='Accuracy',    # takibi doğruluk üzerinden yaptım
    verbose=100
)
model = cb.fit(x_train,y_train,cat_features=cat_features)

0:	learn: 0.3750293	total: 77.9ms	remaining: 1m 17s
100:	learn: 0.5577419	total: 1.71s	remaining: 15.2s
200:	learn: 0.6230967	total: 3.29s	remaining: 13.1s
300:	learn: 0.6959475	total: 5.03s	remaining: 11.7s
400:	learn: 0.7605997	total: 6.84s	remaining: 10.2s
500:	learn: 0.8053408	total: 8.62s	remaining: 8.59s
600:	learn: 0.8329820	total: 10.4s	remaining: 6.91s
700:	learn: 0.8580464	total: 12.2s	remaining: 5.2s
800:	learn: 0.8819396	total: 14s	remaining: 3.47s
900:	learn: 0.9020848	total: 15.8s	remaining: 1.74s
999:	learn: 0.9175451	total: 17.6s	remaining: 0us


In [12]:
model.score(x_test,y_test)

np.float64(0.31514084507042256)

In [13]:
df['readmission_risk'].value_counts()

readmission_risk
Low       1853
Medium    1801
High      1751
Name: count, dtype: int64